In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from math import cos, sin, radians, sqrt

# ----- 尺寸常數 -----
COURT_LEN = 13.4
COURT_W   = 6.1
HALF_W    = COURT_W / 2
NET_X     = 6.7
NET_H     = 1.524

SINGLES_W = 5.18
SINGLES_HALF_W = SINGLES_W / 2  # 2.59 m

SHORT_SERVICE_FROM_NET = 1.98
SHORT_NEAR  = NET_X - SHORT_SERVICE_FROM_NET  # 4.72 m
SHORT_FAR   = NET_X + SHORT_SERVICE_FROM_NET  # 8.68 m

DOUBLES_LONG_FROM_BACK = 0.76
LONG_NEAR = DOUBLES_LONG_FROM_BACK            # 0.76 m
LONG_FAR  = COURT_LEN - DOUBLES_LONG_FROM_BACK# 12.64 m


# ========= 1) 物理模擬：用重力 + 二次阻力（|v|v），Euler 積分 =========
def simulate_trajectory(speed_mps=23.0, yaw_deg=6.0, pitch_deg=20.0,
                        drag_k=0.08, release_height=1.2,
                        dt=0.01, max_t=4.0):
    """
    回傳：
      points: ndarray (N,7) = [x,y,z,vx,vy,vz,t]
      apex:   最高點 dict（含 x,y,z,t）
      cross_net: 過網點 dict（含 x,y,z,t, clearance），可能為 None
      landing: 落地點 dict（含 x,y, z=0, t），可能為 None
    """
    g = 9.81
    yaw   = radians(yaw_deg)
    pitch = radians(pitch_deg)

    # 初速分解
    vx = speed_mps * cos(pitch) * cos(yaw)
    vy = speed_mps * cos(pitch) * sin(yaw)
    vz = speed_mps * sin(pitch)

    x, y, z = 0.0, 0.0, release_height
    t = 0.0

    pts = []
    apex = {"x": x, "y": y, "z": z, "t": t}
    cross_net = None

    prev = None  # 前一個步點（用來檢查穿越）
    landing = None
    while t <= max_t and z >= 0.0:
        pts.append((x, y, z, vx, vy, vz, t))
        if z > apex["z"]:
            apex = {"x": x, "y": y, "z": z, "t": t}

        prev = (x, y, z, vx, vy, vz, t)

        # 阻力 ~ -k |v| v
        vmag = sqrt(vx*vx + vy*vy + vz*vz)
        ax = -drag_k * vmag * vx
        ay = -drag_k * vmag * vy
        az = -g - drag_k * vmag * vz

        # Euler 積分
        vx += ax * dt
        vy += ay * dt
        vz += az * dt
        x  += vx * dt
        y  += vy * dt
        z  += vz * dt
        t  += dt
        
        def lerp(a, b, w):  # 線性內插
            return a + w*(b - a)

        w_net = None
        if (prev[0] - NET_X) * (x - NET_X) <= 0 and x != prev[0]:
            w_net = (NET_X - prev[0]) / (x - prev[0])     # 0~1

        w_land = None
        if prev[2] >= 0.0 and z < 0.0 and z != prev[2]:
            w_land = (0.0 - prev[2]) / (z - prev[2])      # 0~1

        # 僅接受 0<=w<=1
        if w_net is not None and not (0.0 <= w_net <= 1.0):
            w_net = None
        if w_land is not None and not (0.0 <= w_land <= 1.0):
            w_land = None

        # 先算出交點的各項（若有需要）
        def net_point():
            y_net = lerp(prev[1], y, w_net)
            z_net = lerp(prev[2], z, w_net)
            t_net = lerp(prev[6], t, w_net)
            vx_net = lerp(prev[3], vx, w_net)
            vy_net = lerp(prev[4], vy, w_net)
            vz_net = lerp(prev[5], vz, w_net)
            return y_net, z_net, t_net, vx_net, vy_net, vz_net

        def land_point():
            x_land = lerp(prev[0], x, w_land)
            y_land = lerp(prev[1], y, w_land)
            t_land = lerp(prev[6], t, w_land)
            vx_land = lerp(prev[3], vx, w_land)
            vy_land = lerp(prev[4], vy, w_land)
            vz_land = lerp(prev[5], vz, w_land)
            return x_land, y_land, t_land, vx_land, vy_land, vz_land

        # 依先後次序處理事件
        if w_net is not None and w_land is not None:
            # 同一步兩個面都被穿過 → 比較誰先
            if w_land <= w_net:
                # 先落地：只記落地，結束
                x_land, y_land, t_land, vx_land, vy_land, vz_land = land_point()
                landing = {"x": x_land, "y": y_land, "z": 0.0, "t": t_land}
                pts.append((x_land, y_land, 0.0, vx_land, vy_land, vz_land, t_land))
                break
            else:
                # 先到網面：判斷是過網或撞網
                y_net, z_net, t_net, vx_net, vy_net, vz_net = net_point()
                if cross_net is None:
                    cross_net = {"x": NET_X, "y": y_net, "z": z_net,
                                "clearance": z_net - NET_H, "t": t_net}
                # 在網面高度 z_net <= 網高，且 y 在球網寬度內 → 撞網，結束
                if (z_net <= NET_H) and (abs(y_net) <= HALF_W) and (z_net >= 0.0):
                    # 把撞網點補入，讓線停在網面
                    pts.append((NET_X, y_net, z_net, vx_net, vy_net, vz_net, t_net))
                    # 你也可以另外回傳 hit_net 事件（若需要）
                    # hit_net = {"x": NET_X, "y": y_net, "z": z_net, "t": t_net}
                    break
                # 否則只是過網，讓下一步繼續前進

        elif w_land is not None:
            # 只有落地
            x_land, y_land, t_land, vx_land, vy_land, vz_land = land_point()
            landing = {"x": x_land, "y": y_land, "z": 0.0, "t": t_land}
            pts.append((x_land, y_land, 0.0, vx_land, vy_land, vz_land, t_land))
            break

        elif w_net is not None:
            # 只有網面
            y_net, z_net, t_net, vx_net, vy_net, vz_net = net_point()
            if cross_net is None:
                cross_net = {"x": NET_X, "y": y_net, "z": z_net,
                            "clearance": z_net - NET_H, "t": t_net}
            if (z_net <= NET_H) and (abs(y_net) <= HALF_W) and (z_net >= 0.0):
                pts.append((NET_X, y_net, z_net, vx_net, vy_net, vz_net, t_net))
                # hit_net = {"x": NET_X, "y": y_net, "z": z_net, "t": t_net}
                break
        # 兩者皆無 → 繼續迴圈

    points = np.array(pts)
    if landing:
        if not (0 <= landing["x"] <= COURT_LEN and -HALF_W <= landing["y"] <= HALF_W):
            landing = None

    return {"points": points, "apex": apex, "cross_net": cross_net, "landing": landing}


# ========= 2)（你原本的）場地 & 繪圖 =========
def build_figure(sim):
    fig = go.Figure()

    # 球場地板
    fig.add_trace(go.Mesh3d(
        x=[0, COURT_LEN, COURT_LEN, 0],
        y=[-HALF_W, -HALF_W, HALF_W, HALF_W],
        z=[0, 0, 0, 0],
        opacity=0.35, color="seagreen", name="Court",
        i=[0,0], j=[1,2], k=[2,3]
    ))

    def add_line(x1, y1, x2, y2, name, width=6):
        fig.add_trace(go.Scatter3d(
            x=[x1, x2], y=[y1, y2], z=[0, 0],
            mode="lines", line=dict(color="white", width=width), name=name
        ))

    # --- 外框（端線 & 雙打邊線） ---
    add_line(0, -HALF_W, 0,  HALF_W, "baseline (near)")
    add_line(COURT_LEN, -HALF_W, COURT_LEN, HALF_W, "baseline (far)")
    add_line(0, -HALF_W, COURT_LEN, -HALF_W, "sideline (dbl)")
    add_line(0,  HALF_W, COURT_LEN,  HALF_W, "sideline (dbl)")

    # --- 單打邊線 ---
    add_line(0, -SINGLES_HALF_W, COURT_LEN, -SINGLES_HALF_W, "sideline (sgl)")
    add_line(0,  SINGLES_HALF_W, COURT_LEN,  SINGLES_HALF_W, "sideline (sgl)")

    # --- 前發球線（距網 1.98 m）---
    add_line(SHORT_NEAR, -HALF_W, SHORT_NEAR, HALF_W, "short service line (near)")
    add_line(SHORT_FAR,  -HALF_W, SHORT_FAR,  HALF_W, "short service line (far)")

    # --- 中線（從前發球線到各自後方界線）---
    add_line(SHORT_NEAR, 0, 0, 0, "center line (near)")
    add_line(SHORT_FAR, 0, COURT_LEN, 0, "center line (far)")

    # --- 雙打長發球線（距端線 0.76 m）---
    add_line(LONG_NEAR, -HALF_W, LONG_NEAR, HALF_W, "long service line (near)")
    add_line(LONG_FAR,  -HALF_W, LONG_FAR,  HALF_W, "long service line (far)")

    # --- 球網（灰牆）---
    fig.add_trace(go.Mesh3d(
        x=[NET_X, NET_X, NET_X, NET_X],
        y=[-HALF_W, HALF_W, HALF_W, -HALF_W],
        z=[0, 0, NET_H, NET_H],
        opacity=0.5, color="gray", name="Net",
        i=[0,0], j=[1,2], k=[2,3]
    ))

    # ========= 3) 用物理函式取得真實軌跡，取出 x/y/z =========
    pts = sim["points"]                       # ←── [改] 不再用外部 res，改用傳入的 sim
    x, y, z = pts[:,0], pts[:,1], pts[:,2]

    # --- 軌跡線 ---
    fig.add_trace(go.Scatter3d(
        x=x, y=y, z=z, mode="lines",
        line=dict(color="red", width=6), name="Trajectory"
    ))

    # --- （可選）把過網點與落地點標出來 ---
    if sim["cross_net"] is not None:
        cn = sim["cross_net"]
        fig.add_trace(go.Scatter3d(
            x=[cn["x"]], y=[cn["y"]], z=[cn["z"]],
            mode="markers", marker=dict(size=5, color="yellow"),
            name=f'Cross Net (clr={cn["clearance"]:.2f} m)'
        ))

    if sim["landing"] is not None:
        ld = sim["landing"]
        fig.add_trace(go.Scatter3d(
            x=[ld["x"]], y=[ld["y"]], z=[0.0],
            mode="markers", marker=dict(size=5, color="blue"),
            name="Landing"
        ))

    # ========= 4) 視角/比例 =========
    fig.update_layout(
        width=900, height=420,                     # 固定輸出尺寸（避免容器差異）
        margin=dict(l=0, r=0, t=40, b=0),          # 縮小邊距，畫面更滿
        scene=dict(
            xaxis=dict(title="Length (m)", range=[0, COURT_LEN]),
            yaxis=dict(title="Width (m)",  range=[-HALF_W, HALF_W]),
            zaxis=dict(title="Height (m)", range=[0, 5]),
            aspectmode="manual",                    # 手動場景比例
            aspectratio=dict(x=2.2, y=1.0, z=0.35), # 拉長x、壓扁z → 更清楚
            camera=dict(
                eye=dict(x=2.2, y=-2.6, z=0.8),    # 鏡頭位置（可微調）
                center=dict(x=0, y=0, z=0),
                up=dict(x=0, y=0, z=1),
                #projection=dict(type="perspective")
                projection=dict(type="orthographic")# 工程圖感；要透視改 "perspective"
            )
        ),
        showlegend=False,
        uirevision="keep-camera"                    # 更新資料時保留相機狀態
    )
    return fig
#fig.show()

def build_combo_figure(sim):
    pts = sim["points"]
    x, y, z = pts[:,0], pts[:,1], pts[:,2]

    fig = make_subplots(
        rows=1, cols=3,
        specs=[[{"type": "scene"}, {"type": "xy"}, {"type": "xy"}]],
        column_widths=[0.5, 0.25, 0.25],
        subplot_titles=("3D view", "Top view (x–y)", "Side view (x–z)")
    )

    # --- 3D ---
    # 地板（簡化）：也可改成 Mesh3d 球場
    # 球場地板
    fig.add_trace(go.Mesh3d(
        x=[0, COURT_LEN, COURT_LEN, 0],
        y=[-HALF_W, -HALF_W, HALF_W, HALF_W],
        z=[0, 0, 0, 0],
        opacity=0.35, color="seagreen", name="Court",
        i=[0,0], j=[1,2], k=[2,3]
    ), row=1, col=1)

    def add_line(x1, y1, x2, y2, name, width=6):
        fig.add_trace(go.Scatter3d(
            x=[x1, x2], y=[y1, y2], z=[0, 0],
            mode="lines", line=dict(color="white", width=width), name=name
        ), row=1, col=1)

    # --- 外框（端線 & 雙打邊線） ---
    add_line(0, -HALF_W, 0,  HALF_W, "baseline (near)")
    add_line(COURT_LEN, -HALF_W, COURT_LEN, HALF_W, "baseline (far)")
    add_line(0, -HALF_W, COURT_LEN, -HALF_W, "sideline (dbl)")
    add_line(0,  HALF_W, COURT_LEN,  HALF_W, "sideline (dbl)")

    # --- 單打邊線 ---
    add_line(0, -SINGLES_HALF_W, COURT_LEN, -SINGLES_HALF_W, "sideline (sgl)")
    add_line(0,  SINGLES_HALF_W, COURT_LEN,  SINGLES_HALF_W, "sideline (sgl)")

    # --- 前發球線（距網 1.98 m）---
    add_line(SHORT_NEAR, -HALF_W, SHORT_NEAR, HALF_W, "short service line (near)")
    add_line(SHORT_FAR,  -HALF_W, SHORT_FAR,  HALF_W, "short service line (far)")

    # --- 中線（從前發球線到各自後方界線）---
    add_line(SHORT_NEAR, 0, 0, 0, "center line (near)")
    add_line(SHORT_FAR, 0, COURT_LEN, 0, "center line (far)")

    # --- 雙打長發球線（距端線 0.76 m）---
    add_line(LONG_NEAR, -HALF_W, LONG_NEAR, HALF_W, "long service line (near)")
    add_line(LONG_FAR,  -HALF_W, LONG_FAR,  HALF_W, "long service line (far)")

    # --- 球網（灰牆）---
    fig.add_trace(go.Mesh3d(
        x=[NET_X, NET_X, NET_X, NET_X],
        y=[-HALF_W, HALF_W, HALF_W, -HALF_W],
        z=[0, 0, NET_H, NET_H],
        opacity=0.5, color="gray", name="Net",
        i=[0,0], j=[1,2], k=[2,3]
    ), row=1, col=1)

    # --- 軌跡線 ---
    fig.add_trace(go.Scatter3d(
        x=x, y=y, z=z, mode="lines",
        line=dict(color="red", width=6), name="Trajectory"
    ), row=1, col=1)

    # --- （可選）把過網點與落地點標出來 ---
    if sim["cross_net"] is not None:
        cn = sim["cross_net"]
        fig.add_trace(go.Scatter3d(
            x=[cn["x"]], y=[cn["y"]], z=[cn["z"]],
            mode="markers", marker=dict(size=5, color="yellow"),
            name=f'Cross Net (clr={cn["clearance"]:.2f} m)'
        ), row=1, col=1)

    if sim["landing"] is not None:
        ld = sim["landing"]
        fig.add_trace(go.Scatter3d(
            x=[ld["x"]], y=[ld["y"]], z=[0.0],
            mode="markers", marker=dict(size=5, color="blue"),
            name="Landing"
        ), row=1, col=1)

    # ========= 4) 視角/比例 =========
    fig.update_layout(
        width=1200, height=520,                     # 固定輸出尺寸（避免容器差異）
        margin=dict(l=10, r=10, t=40, b=10),          # 縮小邊距，畫面更滿
        scene=dict(
            xaxis=dict(title="Length (m)", range=[0, COURT_LEN]),
            yaxis=dict(title="Width (m)",  range=[-HALF_W, HALF_W]),
            zaxis=dict(title="Height (m)", range=[0, 5]),
            aspectmode="manual",                    # 手動場景比例
            aspectratio=dict(x=2.2, y=1.0, z=0.35), # 拉長x、壓扁z → 更清楚
            camera=dict(
                eye=dict(x=2.2, y=-2.6, z=0.8),    # 鏡頭位置（可微調）
                center=dict(x=0, y=0, z=0),
                up=dict(x=0, y=0, z=1),
                #projection=dict(type="perspective")
                projection=dict(type="orthographic")# 工程圖感；要透視改 "perspective"
            )
        ),
        showlegend=False,
        uirevision="keep-camera"                    # 更新資料時保留相機狀態
    )

    cn = sim.get("cross_net")
    ld = sim.get("landing")
    # --- Top view (x–y) ---
    def add_xy_line(x1, y1, x2, y2, w=2, dash=None, col="#666"):
        fig.add_trace(go.Scatter(x=[x1, x2], y=[y1, y2], mode="lines",
                                 line=dict(width=w, dash=dash, color=col),
                                 hoverinfo="skip", showlegend=False),
                      row=1, col=2)

    # [新增] Top view：完整球場線
    add_xy_line(0, -HALF_W, COURT_LEN, -HALF_W)
    add_xy_line(0,  HALF_W, COURT_LEN,  HALF_W)
    add_xy_line(0, -HALF_W, 0,  HALF_W)
    add_xy_line(COURT_LEN, -HALF_W, COURT_LEN, HALF_W)
    add_xy_line(0, -SINGLES_HALF_W, COURT_LEN, -SINGLES_HALF_W, w=1.5, col="#888")
    add_xy_line(0,  SINGLES_HALF_W, COURT_LEN,  SINGLES_HALF_W, w=1.5, col="#888")
    add_xy_line(SHORT_NEAR, -HALF_W, SHORT_NEAR, HALF_W, w=1.5, col="#888")
    add_xy_line(SHORT_FAR,  -HALF_W, SHORT_FAR,  HALF_W, w=1.5, col="#888")
    add_xy_line(0, 0, SHORT_NEAR, 0, w=1.5, col="#888")
    add_xy_line(SHORT_FAR, 0, COURT_LEN, 0, w=1.5, col="#888")
    add_xy_line(LONG_NEAR, -HALF_W, LONG_NEAR, HALF_W, w=1.5, col="#888")
    add_xy_line(LONG_FAR,  -HALF_W, LONG_FAR,  HALF_W, w=1.5, col="#888")
    add_xy_line(NET_X, -HALF_W, NET_X, HALF_W, w=2, dash="dash", col="#7f7f7f")

    fig.add_trace(go.Scatter(x=x, y=y, mode="lines",
                             line=dict(width=3, color="#d62728"),
                             name="xy"),
                  row=1, col=2)
    if cn:
        fig.add_trace(go.Scatter(x=[cn["x"]], y=[cn["y"]], mode="markers",
                                 marker=dict(size=9, color="#ffa500",
                                             line=dict(width=1, color="#333")),
                                 name="cross"),
                      row=1, col=2)
    if ld:
        fig.add_trace(go.Scatter(x=[ld["x"]], y=[ld["y"]], mode="markers",
                                 marker=dict(size=9, color="#1e90ff",
                                             line=dict(width=1, color="#333")),
                                 name="landing"),
                      row=1, col=2)

    fig.update_xaxes(title_text="x (m)", range=[0, COURT_LEN],
                     zeroline=False, showgrid=False, row=1, col=2)
    fig.update_yaxes(title_text="y (m)", range=[-HALF_W, HALF_W],
                     zeroline=False, showgrid=False,
                     scaleanchor="x2", scaleratio=1, row=1, col=2)
    # --- Side view (x–z) ---
    fig.add_trace(go.Scatter(x=x, y=z, mode="lines",
                             line=dict(width=3, color="#1f77b4"),
                             name="xz"),
                  row=1, col=3)

    if cn:
        fig.add_trace(go.Scatter(x=[cn["x"]], y=[cn["z"]], mode="markers",
                                 marker=dict(size=9, color="#ffa500",
                                             line=dict(width=1, color="#333")),
                                 name="cross"),
                      row=1, col=3)
    if ld:
        fig.add_trace(go.Scatter(x=[ld["x"]], y=[0.03], mode="markers",
                                 marker=dict(size=9, color="#1e90ff",
                                             line=dict(width=1, color="#333")),
                                 name="landing"),
                      row=1, col=3)

    # 地面/網位/網高虛線
    zmax = max(5, float(np.nanmax(z))*1.1 if len(z) else 5)
    fig.add_trace(go.Scatter(x=[0, COURT_LEN], y=[0, 0], mode="lines",
                             line=dict(width=1, color="#aaaaaa"),
                             hoverinfo="skip", showlegend=False),
                  row=1, col=3)
    fig.add_trace(go.Scatter(x=[NET_X, NET_X], y=[0, zmax], mode="lines",
                             line=dict(width=1, dash="dash", color="#aaaaaa"),
                             hoverinfo="skip", showlegend=False),
                  row=1, col=3)
    fig.add_trace(go.Scatter(x=[0, COURT_LEN], y=[NET_H, NET_H], mode="lines",
                             line=dict(width=1, dash="dash", color="#aaaaaa"),
                             hoverinfo="skip", showlegend=False),
                  row=1, col=3)

    fig.update_xaxes(title_text="x (m)", range=[0, COURT_LEN],
                     zeroline=False, showgrid=False, row=1, col=3)
    fig.update_yaxes(title_text="z (m)", range=[0, zmax],
                     zeroline=False, showgrid=False, row=1, col=3)

    # [修正] 統一整張圖樣式
    fig.update_layout(height=520, width=1200,
                      margin=dict(l=10, r=10, t=40, b=10),
                      showlegend=False, uirevision="keep-camera")
    return fig

In [3]:
import time
import json
import paho.mqtt.client as mqtt
import ipywidgets as w
from IPython.display import display

class SimulatedMachineController:
    def __init__(self):
        self.yaw = 0
        self.pitch = 0
        self.speed = 0
        self.spinning = False

    def setYaw(self, yaw: int):
        time.sleep(0.1)
        self.yaw = yaw
        print(f"Simulated: set yaw to {yaw}")

    def getYaw(self) -> int:
        return self.yaw

    def setPitch(self, pitch: int):
        time.sleep(0.1)
        self.pitch = pitch
        print(f"Simulated: set pitch to {pitch}")

    def getPitch(self) -> int:
        return self.pitch

    def setSpeed(self, speed: int):
        time.sleep(0.1)
        self.speed = speed
        print(f"Simulated: set speed to {speed}")

    def getSpeed(self) -> int:
        return self.speed

    def getAllSettings(self):
        return self.getYaw(), self.getPitch(), self.getSpeed()

    def startSpinning(self):
        self.spinning = True
        print("Simulated: started spinning")

    def stopSpinning(self):
        self.spinning = False
        print("Simulated: stopped spinning")

    def detectBall(self) -> bool:
        print("Simulated: detecting ball -> True")
        return True

    def serve(self) -> bool:
        print("Simulated: starting serve sequence")
        if not self.detectBall():
            self.startSpinning()
            time.sleep(1)
            self.stopSpinning()
            print("Simulated: no ball detected, serve failed")
            return False
        print("Simulated: serving ball")
        time.sleep(0.5)
        return True


class MQTTSimulator:
    def __init__(self, broker='broker.emqx.io', port=1883,
                 command_topic='Machine_A', status_topic='app'):
        self.broker = broker
        self.port = port
        self.command_topic = command_topic
        self.status_topic = status_topic
        self.client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
        self.controller = SimulatedMachineController()
        self.is_serving = False
        self.is_stopped = False

    def on_connect(self, client, userdata, flags, rc, properties):
        print("Simulator connected to MQTT broker with result code", rc)
        # 訂閱命令主題與廣播主題
        client.subscribe(self.command_topic)
        client.subscribe("broadcast")
    

    def on_message(self, client, userdata, msg):
        payload = msg.payload.decode()
        print(payload)
        # 如果訊息來自 "broadcast" 主題，檢查是否為查詢機器名稱的訊息
        if msg.topic == "broadcast":
            try:
                data = json.loads(payload)
                if data.get("msg_type") == "query" and data.get("parameter") == "topic_name":
                    reply = {
                        "source": "Machine_A",
                        "msg_type": "reply",
                        "parameter": {
                            "topic_name": "Machine_A",
                            "range": {
                                "speed": {"min": 0, "max": 100},
                                "yaw": {"min": -90, "max": 90},
                                "pitch": {"min": -45, "max": 45}
                            }
                        }
                    }
                    # 將回覆發佈到 "app" 主題，讓 MachineClient 收到
                    self.client.publish(self.status_topic, json.dumps(reply))
                    print("Simulator replied with topic name and range info")
                    return
            except Exception as e:
                print("Error parsing broadcast JSON:", e)
        # 如果訊息來自命令主題，先嘗試解析 JSON，看是否為查詢狀態
        if msg.topic == self.command_topic:
            try:
                data = json.loads(payload)
                if data.get("msg_type") == "query":
                    if data.get("parameter") == "status":
                        # 回覆狀態資訊
                        status_reply = {
                            "source": "Machine_A",
                            "msg_type": "reply",
                            "parameter": {
                                "status": {
                                    "available": True,
                                    "settings": {
                                        "speed": self.controller.getSpeed(),
                                        "yaw": self.controller.getYaw(),
                                        "pitch": self.controller.getPitch()
                                    },
                                    "range": {
                                        "speed": {"min": 0, "max": 100},
                                        "yaw": {"min": -90, "max": 90},
                                        "pitch": {"min": -45, "max": 45}
                                    }
                                }
                            }
                        }
                        self.client.publish(self.status_topic, json.dumps(status_reply))
                        print("Simulator replied with status info")
                        return
                elif data.get("msg_type") == "command":
                    commands = data.get("parameter").split(';')
                    for command in commands:
                        if '=' in command:
                            key, value = command.split('=')
                            self.process_command(key.strip(), value.strip())
                        elif command:
                            self.process_command(command.strip(), None)
                    
            except Exception as e:
                print("Error parsing JSON for query or commands:", e)
        else:
            # 如果不是來自廣播或命令主題，則直接印出訊息
            print("Simulator received message on unknown topic:", msg.topic, payload)


    def process_command(self, command, value):
        if command == 'speed':
            try:
                speed = int(value)
                self.controller.setSpeed(speed)
                self.publish_status()
            except Exception as e:
                print("Error setting speed:", e)
        elif command == 'yaw':
            try:
                yaw = int(value)
                self.controller.setYaw(yaw)
                self.publish_status()
            except Exception as e:
                print("Error setting yaw:", e)
        elif command == 'pitch':
            try:
                pitch = int(value)
                self.controller.setPitch(pitch)
                self.publish_status()
            except Exception as e:
                print("Error setting pitch:", e)
        elif command == 'serve':
            self.is_serving = True
            result = self.controller.serve()
            self.publish_status()
            if result:
                self.client.publish(self.status_topic, "serve=done")
            else:
                self.client.publish(self.status_topic, "serve=failed")
        elif command == 'MQTT_disconnect':
            print("Simulator: received MQTT_disconnect command")
            self.stop()
        else:
            print("Simulator: set", command)

    def publish_status(self):
        yaw, pitch, speed = self.controller.getAllSettings()
        message = f"speed={speed};yaw={yaw};pitch={pitch}"
        print("Simulator publishing status:", message)
        self.client.publish(self.status_topic, message)

    def start(self):
        self.client.on_connect = self.on_connect
        self.client.on_message = self.on_message
        self.client.connect(self.broker, self.port, 60)
        self.client.loop_start()
    def stop(self):
        print("\nShutting down simulator...")
        self.client.unsubscribe(self.command_topic)
        self.client.unsubscribe("broadcast")
        self.client.disconnect()
        self.client.loop_stop()
        print("Simulator disconnected.")
        self.is_stopped = True


if __name__ == '__main__':
    simulator = MQTTSimulator(command_topic='Machine_A',
                            status_topic='abcde12345')
    dragk = w.FloatSlider(description="drag k",min=0, max=0.2, step=0.005, value=0.08)
    h0    = w.FloatSlider(description="h0",    min=0.5, max=2.2, step=0.01, value=1.2)
    speed_label = w.Label(value=f"Speed: {simulator.controller.getSpeed()}")
    yaw_label = w.Label(value=f"Yaw: {simulator.controller.getYaw()}°")
    pitch_label = w.Label(value=f"Pitch: {simulator.controller.getPitch()}°")

    btn = w.Button(description='更新', button_style='info')

    out = w.Output()
    def redraw(*_):
        speed_val = simulator.controller.getSpeed()
        yaw_val = simulator.controller.getYaw()
        pitch_val = simulator.controller.getPitch()
        
        speed_label.value = f"Speed: {speed_val} m/s"
        yaw_label.value = f"Yaw: {yaw_val}°"
        pitch_label.value = f"Pitch: {pitch_val}°"
        
        sim = simulate_trajectory(speed_mps=speed_val, yaw_deg=yaw_val,
                                    pitch_deg=pitch_val, drag_k=dragk.value,
                                    release_height=h0.value, dt=0.01, max_t=3.0)
        fig = build_combo_figure(sim)
        with out:
            out.clear_output(wait=True)
            fig.show()
    simulator.start()
    try:
        for s in (dragk,h0):
            s.observe(redraw, names="value")
        btn.on_click(redraw)
        display(w.VBox([speed_label, yaw_label, pitch_label, dragk, h0, btn, out]))
        redraw()
        while True:
            if simulator.is_serving:
                redraw()
                simulator.is_serving = False
            elif simulator.is_stopped:
                print("Simulator terminated.")
                break
            time.sleep(1)
    except KeyboardInterrupt:
        simulator.stop()
        print("Simulator terminated.")

Simulator connected to MQTT broker with result code Success
{"msg_type": "query", "parameter": "topic_name"}
Simulator replied with topic name and range info
{"msg_type": "command", "parameter": "MQTT_disconnect"}
Simulator: received MQTT_disconnect command

Shutting down simulator...
Simulator disconnected.
Simulator terminated.
